### 数据导入

In [16]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scorecardpy as sc
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, roc_curve
import pickle

In [17]:
df_origin = pd.read_excel('../data/default_credit_card.xls', header=1)
df_origin.head(5)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
#删除ID列
df = df_origin.drop("ID", axis=1)
#重命名目标变量
df.rename(columns={"default payment next month":"target"}, inplace=True)

In [4]:
#确认是否有缺失值
all(df.isnull().sum())==0

True

In [5]:
#违约分布（目标变量）统计：
df['target'].value_counts()

target
0    23364
1     6636
Name: count, dtype: int64

### 预处理

In [6]:
#分箱，并筛选有效特征
bins = sc.woebin(df, y="target")
iv_summary = sc.iv(df, y="target")

[INFO] creating woe binning ...


/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version 

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:361: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  init_bin = init_bin.groupby('brkp', group_keys=False).agg({
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:361: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  init_bin = init_bin.groupby('brkp', group_keys=False).agg({
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:361: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be us

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({'good':sum, 'bad':sum}).reset_index()\
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({'good':sum, 'bad':sum}).reset_index()\
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current b

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({'good':sum, 'bad':sum}).reset_index()\
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({'good':sum, 'bad':sum}).reset_index()\
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current b

...

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({'good':sum, 'bad':sum}).reset_index()\
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({'good':sum, 'bad':sum}).reset_index()\
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:425: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation.

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:361: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  init_bin = init_bin.groupby('brkp', group_keys=False).agg({
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({'good':sum, 'bad':sum}).reset_index()\
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:320: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  init_bin = dtm.groupby('bin', group_keys=False)['y'].agg([n0, n1])\
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg({'good':sum, 'bad':sum}).reset_index()\
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/woebin.py:414: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/info_value.py:80: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(goodbad) \
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/info_value.py:80: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(goodbad) \
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/info_value.py:80: FutureWarning: DataFrameGroupBy

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/info_value.py:80: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(goodbad) \
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/info_value.py:80: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(goodbad) \
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/info_value.py:80: FutureWarning: DataFrameGroupBy

In [7]:
print(iv_summary)

     variable  info_value
3       PAY_0    0.877161
11      PAY_2    0.548882
22      PAY_3    0.416046
0       PAY_4    0.367266
1    PAY_AMT1    0.354998
18   PAY_AMT6    0.340639
15      PAY_5    0.340341
12   PAY_AMT3    0.340153
16   PAY_AMT4    0.339778
5    PAY_AMT5    0.339478
21   PAY_AMT2    0.330309
17      PAY_6    0.293846
20  BILL_AMT6    0.204642
13  BILL_AMT5    0.195339
10  LIMIT_BAL    0.192544
6   BILL_AMT4    0.184890
9   BILL_AMT3    0.172999
14  BILL_AMT2    0.164650
2   BILL_AMT1    0.151770
19  EDUCATION    0.038362
8         AGE    0.029208
4         SEX    0.009180
7    MARRIAGE    0.007235


In [8]:
#筛选出区分力>0.2的特征值
good_vars = iv_summary[iv_summary["info_value"]>0.02]["variable"].tolist()
df_filter = df[good_vars + ["target"]]

In [9]:
#将保留的特征转为WOE编码，得到建模专用数据集
df_woe = sc.woebin_ply(df_filter, bins)

[INFO] converting into woe values ...


/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: Us

In [25]:
#将数据集分为train和test
train, test = train_test_split(df_woe, test_size=0.3, random_state=42, stratify=df_woe["target"])

In [27]:
#导出为csv格式
train.to_csv('../data/train_woe.csv', index=False)
test.to_csv('../data/test_woe.csv', index=False)

In [22]:
with open("bins.pkl", "wb") as f:
    pickle.dump(bins, f)

In [24]:
with open("good_vars.pkl", "wb") as f:
    pickle.dump(good_vars, f)